# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

meta = dataset.metadata
print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Note:** All references use the Croissant `@id` for unambiguous identification.

In [ ]:
# List all available record sets and their fields

print("Available record sets (by @id):")
for rs in dataset.record_sets:
    print(f"- @id: {rs['@id']}, name: {rs.get('name', 'N/A')}")

# Now, for each record set, print out its fields/columns
for rs in dataset.record_sets:
    print(f"\nRecord Set '@id': {rs['@id']}")
    field_ids = [field['@id'] for field in rs.get('field', [])]  # field -> list of field objects
    if field_ids:
        print("  Fields (by @id):")
        for field in rs['field']:
            print(f"    - {field['@id']}, name: {field.get('name', 'N/A')}")
    else:
        print("  No fields found.")# For a quick example of record structure, preview one row from each record set using the mlcroissant API
for rs in dataset.record_sets:
    print(f"\nPreviewing first record in record set '@id': {rs['@id']}")
    try:
        records_preview = list(dataset.records(record_set=rs['@id']))
        if records_preview:
            print(records_preview[0])
        else:
            print("  No records available.")
    except Exception as e:
        print(f"  Error loading records: {e}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# List all record sets by @id
record_sets = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
    except Exception as e:
        print(f"Skipping {record_set_id} due to error: {e}")

print("Extracted DataFrames for the following record set @ids:")
for k in dataframes.keys():
    print("-", k)

# Choose a record set for further exploration (pick the first if unsure)
target_record_set = record_sets[0] if record_sets else None
if target_record_set and target_record_set in dataframes:
    print(f"\nColumns in record set {target_record_set}:")
    print(dataframes[target_record_set].columns.tolist())
    display(dataframes[target_record_set].head())
else:
    print("No dataframes available to display.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

Be sure to refer to columns/fields via their Croissant `@id`.

In [ ]:
import numpy as np

if not target_record_set or target_record_set not in dataframes:
    print("No valid record set found for EDA.")
else:
    df = dataframes[target_record_set]
    # Try to automatically select a numeric field (by inspecting dtypes or by column name patterns)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_cols:
        # Try to convert columns that look like numbers
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='ignore')
            except Exception:
                continue
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if numeric_cols:
        numeric_field = numeric_cols[0]
        # Example: filter records where the numeric field > threshold
        threshold = df[numeric_field].mean() if df[numeric_field].notnull().any() else 0
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to group by a categorical/text field
        group_field = None
        for col in df.columns:
            if col != numeric_field and df[col].dtype == 'object':
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"\nGrouped data by {group_field} (mean of {numeric_field}):")
            print(grouped_df.head())
    else:
        print("No numeric columns found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

if not target_record_set or target_record_set not in dataframes:
    print("No valid dataframe for visualization.")
else:
    df = dataframes[target_record_set]
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        plt.figure(figsize=(8, 5))
        df[numeric_cols[0]].hist(bins=30)
        plt.title(f"Distribution of {numeric_cols[0]}")
        plt.xlabel(numeric_cols[0])
        plt.ylabel("Count")
        plt.show()

        if len(numeric_cols) > 1:
            plt.figure(figsize=(7, 7))
            plt.scatter(df[numeric_cols[0]], df[numeric_cols[1]])
            plt.title(f"{numeric_cols[0]} vs {numeric_cols[1]}")
            plt.xlabel(numeric_cols[0])
            plt.ylabel(numeric_cols[1])
            plt.show()
    else:
        print("No numeric columns available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR\textsuperscript{2} Croissant dataset was loaded and explored using field and record set `@id`s,
- Numeric and categorical summaries can be produced for each record set,
- Automated EDA can identify candidate statistical groupings and normalization.

For workflow automation, extend this notebook to select target columns and record sets by their `@id` as shown above. See the metadata and cell outputs for `@id` discovery.

**References:**
- [mlcroissant documentation](https://mlcommons.github.io/croissant/)
- [Dataset Source](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

Happy data exploring!